# An OpenAI Agents SDK agent, trading a market that never existed

**The OpenAI Agents SDK** is the agent framework. **Tradefloor** is the
simulated market that measures what the agent does.

The agent below is an ordinary `agents.Agent`: a name, instructions, a model.
Nothing about it is Tradefloor-specific, and the adapter does not rebuild it
-- it binds a decision contract onto a *clone* and sends the observation as a
message, so the object you would run anywhere else is the object being
measured here.

What this notebook shows, in order: what the model is actually sent, what it
decided and why, how the wording of its instructions changed how far it
missed a limit by, and what it was never allowed to see.

A backtest on real data cannot say any of that. The model has read the whole
internet, and it has not read this market, because this market was generated
rather than recorded.

## 1. Live calls, and the recording

Tradefloor's market is deterministic. A language model behind an API is not,
so two live runs give two different agents and neither is reproducible.

This notebook **replays a recording by default and calls the model only when
asked to**. Three conditions gate a live run, and the opt-in comes first:
`TRADEFLOOR_LIVE_EXAMPLES=1`, then the API key, then the SDK. A credential
sitting in the environment is not consent to spend it -- without that gate,
anyone with a key exported who ran the slow test suite would re-execute every
notebook live and pay for it, having asked for neither.

The market re-executes for real either way; only the agent's answers come
from the recording, keyed by a digest of the exact input it was sent.

The committed recording is a genuine run. Nothing in it was written by hand.

In [1]:
import json
import sys
from pathlib import Path

import tradefloor as tf
from tradefloor.integrations.common import Transcript

# The production adapter. This notebook implements none of the integration:
# `tradefloor/integrations/openai_agents.py` is the source of truth, and this
# is the same object the shipped example and the test suite use.
from tradefloor.integrations.openai_agents import (BRIEF, OpenAIAgentsAdapter,
                                                   payload_of)

# Constants come from the example script beside this notebook, for the same
# reason: one definition, so the two cannot drift into describing different
# experiments.
sys.path.insert(0, str(Path.cwd()))
import five_days as example

LIVE = example.can_run_live()

print(f"tradefloor {tf.__version__}")
print(f"mode        {'live, calling the model' if LIVE else 'replay'}")
if LIVE:
    # The ceiling, printed BEFORE the run cell spends anything. One turn is
    # one model call, and the budget is a constant so this estimate and the
    # call it describes cannot drift apart.
    print(f"            {example.LIVE_MODEL}, {example.DAYS} decisions, "
          f"up to {example.LIVE_MAX_TURNS} model calls each")
    print(f"            ceiling {example.DAYS * example.LIVE_MAX_TURNS} "
          f"calls; this spends real money")
else:
    print(f"            live would need "
          f"{', '.join(example.missing_for_live())}")

tradefloor 0.8.5
mode        replay
            live would need TRADEFLOOR_LIVE_EXAMPLES=1, OPENAI_API_KEY


## 2. The market

Four synthetic instruments. The tickers, the sectors and every fundamental
are generated, so nothing the model knows about real listed companies applies
-- which is the point. An agent cannot recall this market's history, because
it does not have one.

In [2]:
roster = example.universe()
for instrument in roster:
    print(f"{instrument.ticker:10} {instrument.sector:20} "
          f"${instrument.initial_price:7.2f}  "
          f"growth {instrument.revenue_growth:5.0%}  "
          f"ADV {instrument.avg_volume:,.0f}")
print(f"\nseed {example.SEED}, {example.DAYS} days, "
      f"{len(roster)} instruments")

TECH_A     technology           $ 140.00  growth   30%  ADV 5,000,000
TECH_B     technology           $  95.00  growth   22%  ADV 5,000,000
BANK_A     financial_services   $  60.00  growth    5%  ADV 5,000,000
STAPLE_A   consumer_staples     $  48.00  growth    2%  ADV 5,000,000

seed 4242, 5 days, 4 instruments


## 3. The agent

An `agents.Agent` and nothing more. The adapter takes it positionally, the
way every adapter in this package takes its framework object.

The one change the adapter makes is to bind Tradefloor's decision contract as
the agent's `output_type`, and it does that on `Agent.clone(...)` -- so the
original keeps whatever output type it had, which here is none at all.

The standing brief is sent as a *message*, never written into `instructions`.
Overwriting somebody's system prompt would evaluate a different agent from
the one they wrote.

In [3]:
if LIVE:
    from agents import Agent

    pm = Agent(
        name="Portfolio Manager",
        instructions=(
            "You manage a concentrated equity book in a simulated market. "
            "You prefer buying weakness and trimming strength, you size "
            "positions against the limits the observation states, and you "
            "would rather do nothing than force a trade."
        ),
        model=example.LIVE_MODEL,
    )
    agent = OpenAIAgentsAdapter(pm, mode="live", recorder=Transcript(),
                                max_turns=example.LIVE_MAX_TURNS, arm="live")
    agent.recorder.meta.update(agent.provenance())
    print(f"live: {pm.name} on {example.LIVE_MODEL}")
else:
    pm = None
    transcript = Transcript.load(example.FIXTURE)
    agent = OpenAIAgentsAdapter(mode="replay", transcript=transcript,
                                arm="replay")
    print(f"replaying {len(transcript)} recorded interactions from")
    print(f"  {example.FIXTURE.name}\n")
    for field in ("framework", "framework_version", "framework_url",
                  "entry_point", "provider", "model", "agent_name", "mode",
                  "generation", "decision_every_steps",
                  "decision_schema_version", "instructions_version",
                  "instructions_digest", "recorded_utc"):
        print(f"  {field:<25} {transcript.meta.get(field)}")

replaying 5 recorded interactions from
  five-days.json

  framework                 openai-agents
  framework_version         0.22.3
  framework_url             https://github.com/openai/openai-agents-python
  entry_point               agents.Runner.run
  provider                  openai
  model                     gpt-5.2
  agent_name                Portfolio Manager
  mode                      live
  generation                {'max_turns': 6, 'tracing': False}
  decision_every_steps      6
  decision_schema_version   1
  instructions_version      1
  instructions_digest       ca0015e5c92204a3
  recorded_utc              2026-09-26T10:29:25+00:00


Every field a replayed run needs to describe itself is populated: what
framework, what version, what entry point, what model, what generation
settings, and digests of the instructions. Nowhere in that structure is
there a place to put a credential, which is deliberate -- this dictionary is
printed and written into artifacts.

## 4. What the model is actually sent

This is the cell worth reading twice. `payload_of` reads the serialized
observation back out of the exact call the adapter made, so what follows is
not a reconstruction -- it is the input, recovered from the recording.

Two messages go to the model: the standing brief, and the observation as
JSON. Everything the agent knows about this market is below.

In [4]:
first = agent.transcript.entries[0] if not LIVE else None
sample = payload_of(first["prompt"]) if first else None

if sample is None:
    print("live run: the payload is built at the first decision point")
else:
    print("MESSAGE 1 -- the standing brief\n")
    print(BRIEF)
    print("MESSAGE 2 -- the observation, in full\n")
    print(json.dumps(sample, indent=2)[:2600])

MESSAGE 1 -- the standing brief

You are trading a simulated market. Every instrument is synthetic: the tickers, the sectors and the fundamentals are generated, so anything you know about real listed companies does not apply here.

The message after this one is the entire observation, as JSON. You have no other data source and no browsing. Where a field is null it is genuinely unknown -- a five-day return is null until five days have been observed -- and null is not zero.

Seek attractive risk-adjusted returns while controlling downside risk. You may buy, sell, resize or maintain positions, and you are not required to trade: leaving the book alone is a decision.

Two separate limits bound your size and you must respect both. `max_order_shares` is what this market can absorb in one order without your own trade moving the price against you; a larger request is clipped, and the clip is recorded against you rather than silently applied. `portfolio.buying_power` is the additional gross noti

Note what the observation carries and what it does not. Prices, the top of
book, average volume, the agent's own position, the macro figures as published (the
`cycle` is the phase as announced, which lags the true one), and **two
different size limits**. No fair value, no factor attribution, no mispricing,
no future macro path. Section 8 checks that rather than trusting it.

## 5. The run

The market advances every step; the agent is asked once a day. On the steps
in between, the adapter records the prices it saw and returns nothing -- a
manager watches the book continuously and revisits it on a schedule.

In [5]:
scores = tf.evaluate({"pm": agent}, seed=example.SEED, universe=roster,
                     days=example.DAYS)
card = scores["pm"]

print(f"decisions   {len(agent.record)}")
print(f"trades      {card.trades}")
print(f"rejected    {card.rejected}")
print(f"turnover    {card.turnover:,.0f}")
print(f"pnl         {card.pnl:+,.0f}")
print(f"return      {card.return_pct:+.2f}%")
print(f"impact      {card.impact_bps:+.2f} bps")

decisions   5
trades      3
rejected    0
turnover    2,825,350
pnl         +57,576
return      +5.76%
impact      +0.28 bps


In [6]:
# The agent's own object is untouched: the contract was bound on a clone.
if LIVE:
    print(f"pm.output_type after the run: {pm.output_type}")
else:
    print("replay: no agent object was involved, which is the point --")
    print("a recorded run needs no key, no network and no SDK install.")
    print(f"'agents' imported: {'agents' in sys.modules}")

replay: no agent object was involved, which is the point --
a recorded run needs no key, no network and no SDK install.
'agents' imported: False


## 6. What it decided, and why

The rationale is the model's own, recorded verbatim. It never affects
execution -- only the actions do -- but it is the thing a backtest cannot
give you: a statement of reasoning you can check against what the market
actually did.

In [7]:
for entry in agent.record:
    acted = ", ".join(f"{a['side']} {a['quantity']:,.0f} {a['symbol']}"
                      for a in entry["decision"]["actions"]
                      if a["side"] != "HOLD") or "no change"
    print(f"day {entry['day']}  {acted}")
    print(f"        \"{entry['decision']['rationale']}\"")
    if entry["clipped"]:
        print(f"        clipped: {entry['clipped']}")
    print()

day 0  no change
        "Day 0 has no return/volatility/fundamental information to anchor expected return or risk, so initiating positions would be arbitrary. With ample buying power available, the best risk-controlled choice is to wait for at least a few observations before sizing into weakness or trimming strength."

day 1  SELL 0 TECH_A, BUY 15,000 TECH_B, SELL 0 BANK_A, SELL 0 STAPLE_A
        "Prefer buying weakness: TECH_B is the only clear down mover on the day with comparable volatility, so initiate a moderate starter position. Avoid chasing strength in TECH_A and keep the rest unchanged to preserve buying power and limit downside."

day 2  SELL 5,000 TECH_B
        "TECH_B is up strongly on the day; trim strength to reduce leverage and rebuild buying power. No clear weakness elsewhere to add risk today given limited buying power."

day 3  no change
        "No clear weakness to buy: all names have positive 1d and 5d momentum, and our only position (TECH_B) is only modestly up

## 7. Two size limits, and which one binds

An observation that names only one size limit is a trap, and this is the
concrete, teachable property of the environment.

- **`max_order_shares`** is the *participation* cap: how much of one name the
  MARKET can absorb in a single order without the order itself moving the
  price. It is 2% of average daily volume.
- **`portfolio.buying_power`** is the *funding* cap: how much additional
  gross notional this BOOK can carry before the leverage limit refuses the
  trade.

They are unrelated numbers, and **which one binds is a property of the book,
not a rule**. The participation cap is fixed by trading volume; buying power
scales with equity. So a small book against liquid names is limited by what
it can fund, and a large book against thin names is limited by what the
market can absorb. Both are stated precisely so the agent does not have to
guess which regime it is in.

The payload used to state only the participation cap, and four independent
agents -- two frontier models among them -- sized to it, were refused at a
limit the observation never mentioned, and scored zero trades.

The arithmetic below is computed from the committed observation rather than
quoted, including the equity at which the two limits swap places:

In [8]:
pf = sample["portfolio"] if sample else None
if pf is None:
    print("live run: rerun in replay mode to see the recorded arithmetic")
else:
    participation = sum(a["max_order_shares"] * a["price"]
                        for a in sample["assets"])
    print(f"equity                        {pf['net_worth']:>14,.0f}")
    print(f"leverage cap                  {pf['max_leverage']:>14.1f}x")
    print(f"buying power (funding cap)    {pf['buying_power']:>14,.0f}")
    print(f"max_order_shares, all four    {participation:>14,.0f}"
          f"   <- participation cap")
    print()
    # Which limit binds is derived, never asserted: on a bigger book the
    # inequality flips, and prose that assumed one answer would contradict
    # the numbers printed directly above it.
    crossover = participation / pf["max_leverage"]
    funding_binds = participation > pf["buying_power"]
    print(f"Sizing every name to its participation cap asks for "
          f"{participation / pf['net_worth']:.1f}x equity, against a "
          f"{pf['max_leverage']:.0f}x limit.")
    print(f"On THIS book the binding limit is "
          f"{'buying power' if funding_binds else 'the participation cap'}.")
    print(f"They swap at about {crossover:,.0f} of equity: below that the "
          f"funding cap binds,")
    print("above it the market's capacity does. Neither is the rule; the "
          "book decides.")

equity                             1,000,000
leverage cap                             2.0x
buying power (funding cap)         2,000,000
max_order_shares, all four        34,300,000   <- participation cap

Sizing every name to its participation cap asks for 34.3x equity, against a 2x limit.
On THIS book the binding limit is buying power.
They swap at about 17,150,000 of equity: below that the funding cap binds,
above it the market's capacity does. Neither is the rule; the book decides.


The brief is the other half of the trap, and this notebook has now been
recorded six times against `gpt-5.2` on this same seed and roster, three of
them while that brief changed. The results are worth setting out honestly,
because they do not all point the same way.

| brief says | trades | rejected | worst overshoot |
|---|---|---|---|
| size against `max_order_shares` only | 8 | 1 | **3.02x** vs 2.00x -- 51% over |
| both limits, "the funding limit is usually the binding one" | 8 / 10 | 1 / 0 | 2.01x -- 0.5% over |
| both limits, no claim about which binds (0.8.x, pt-v19) | 3 | 1 | 2.19x -- 9.3% over |
| both limits, no claim about which binds (0.8.5, pt-v19) | 10 | 0 | none -- peak 1.74x |
| both limits, no claim about which binds (0.8.5, an earlier pt-v20) | 1 | 2 | **3.26x** vs 2.00x -- 63% over |
| both limits, no claim about which binds *(committed, 0.8.5, pt-v20)* | 3 | 0 | none -- peak 1.86x |

The committed run stayed inside the limit. The run before it, on an
earlier pt-v20 vector, is the largest miss in the table. Together they undo
what this section used to call robust: that naming only the participation
cap produced an order half again the limit, while naming both kept every
later run inside 10%. With both limits named, four runs of the same brief
have landed 9.3% over, inside, 63% over and inside. The brief does not
settle how the model sizes; one run is one draw of it.

The two hint rows are one sample each too: one overshot by 0.5% and one not
at all. That is enough to see that the model can miss by a lot whatever the
brief says, and nowhere near enough to attribute a difference to the
wording.

The last three rows were recorded as 0.8.5 changed the market: an agent's
orders now reach it once, on the minute after the fill, rather than on
every minute of the step; the default preset moved to pt-v20; and pt-v20
moved to the vector its grade passed on. Each change moves the prices the
model is shown, so each needed a new recording.

**The hint was removed anyway, and for a reason that is not about
performance.** "The funding limit is usually the binding one" is false on a
larger book: the participation cap is fixed by volume, buying power scales
with equity, and above the crossover printed above the inequality flips. A
harness that ships a false generalisation because it happens to nudge the
model helpfully on the roster where it is true is shaping the behaviour it
then measures, which is the one thing this environment exists not to do. The
model is given both numbers and can determine which binds.

In [9]:
print(f"{'day':>3} {'buying power':>14} {'bought':>13} {'sold':>12} "
      f"{'net':>13} {'headroom':>12}")
for entry in agent.record:
    p = payload_of(entry["prompt"])
    price = {a["symbol"]: a["price"] for a in p["assets"]}
    buys = sum(a["quantity"] * price[a["symbol"]]
               for a in entry["decision"]["actions"] if a["side"] == "BUY")
    sells = sum(a["quantity"] * price[a["symbol"]]
                for a in entry["decision"]["actions"] if a["side"] == "SELL")
    power, net = p["portfolio"]["buying_power"], buys - sells
    print(f"{entry['day']:>3} {power:>14,.0f} {buys:>13,.0f} {sells:>12,.0f} "
          f"{net:>13,.0f} {power - net:>12,.0f}"
          f"{'   <-- over' if net > power else ''}")

print()
print(f"rejected orders: {card.rejected}")
for err in card.errors:
    print(f"  {err}")

day   buying power        bought         sold           net     headroom
  0      2,000,000             0            0             0    2,000,000
  1      2,000,000     1,400,229            0     1,400,229      599,771
  2        629,030             0      476,710      -476,710    1,105,739
  3      1,110,825             0            0             0    1,110,825
  4      1,100,076       948,176            0       948,176      151,900

rejected orders: 0


No row went over. The model held on day 0 for want of history. On day 1
it bought 15,000 `TECH_B`, the only name down on the day, for 1,400,229 of
notional on a book with 1,000,000 of equity and 2,000,000 of buying power:
gross exposure of about 1.39x. The same answer carried a `SELL` of zero
shares for each of the other three names, which it did not hold; a zero
quantity places no order. On day 2 `TECH_B` was up 2.1% on the day and it
sold 5,000, "to reduce leverage and rebuild buying power". On day 3 every
name was up on the day and over five days, and it held. On day 4 `TECH_B`
was down 1.1% and it bought 10,000 back: 948,176 against 1,100,076 of
buying power, leaving 151,900 of headroom. Leverage peaked at 1.86x.

Its day 4 rationale says the order "remains well within max_order_shares
and available buying power", and this time both halves were true. The run
before this one said the same of a plan that asked for 3.26x equity, and
the market refused two of its three orders. The sentence is the same; what
it describes is not, so the table above counts what was placed, not what
the rationale claims.

`buying_power` is also a snapshot taken when the observation is built. The
orders execute afterwards against a book whose prices have moved, so even an
agent that sizes exactly to the stated cap can cross it -- which is why the
market checks leverage at execution rather than trusting the plan.

A refusal is the *market's*, not the adapter's, and it names the number.
The decision is recorded in full either way, so the trace shows what the
agent asked for beside what it got -- an experiment that silently dropped
the unaffordable leg would score a plan the agent never made, and a refused
order is not the same thing as a considered decision to hold.

## 8. What it was never shown

The simulator knows the answer key: each instrument's fair value, how far
the price sits from it (`mispricing_s`), the factor attribution of every
price move, and the macro path the run has not reached yet. An agent reading
any of it inverts the simulator, and the experiment around it measures
nothing.

Since 0.8.5 an agent's `Observation.engine` is a read-only market view that
refuses all of that, and the adapter's observation mapping is an
**allowlist, written out field by field**, built from that view.

Below, the answer key is read from the simulator itself. A watcher replays
the same recording and declares `privileged = True`, the one route to
hidden state, which the harness records on its scorecard as
`uses_hidden_state`. At each decision it writes down the fair value and the
mispricing the engine held. Every number in every recorded prompt is then
checked against them.

In [10]:
import math
import struct


class Witness:
    """Watches the replay and writes down what only the simulator knew.

    `privileged = True` is the declared way to read hidden state: the
    harness hands this agent `obs.hidden` and its scorecard says
    `uses_hidden_state=True`. The adapter inside it is not privileged. It
    gets the same read-only view as every other agent, and only it talks
    to the model.
    """

    privileged = True

    def __init__(self, inner):
        self.inner = inner
        self.key = []

    def act(self, obs):
        if obs.is_first_step_of_day:
            s = struct.unpack(f"<{len(obs.tickers)}d",
                              obs.hidden.column("mispricing_s"))
            # The engine's prices are `fair_value * exp(s)`, so the fair
            # value is the price with the mispricing taken out.
            self.key.append({t: (obs.price(t) * math.exp(-si), si)
                             for t, si in zip(obs.tickers, s)})
        return self.inner.act(obs)


# The run above, replayed: from the committed file, or from what a live run
# just recorded.
recording = agent.recorder if LIVE else Transcript.load(example.FIXTURE)
witness = Witness(OpenAIAgentsAdapter(mode="replay", transcript=recording))
watched = tf.evaluate({"pm": witness}, seed=example.SEED,
                      universe=example.universe(), days=example.DAYS)["pm"]
# Watching changed nothing: the same decisions, the same fills.
assert watched.pnl == card.pnl and watched.uses_hidden_state

# Every number the model was sent, from every recorded call.
def numbers(node):
    if isinstance(node, dict):
        for v in node.values():
            yield from numbers(v)
    elif isinstance(node, list):
        for v in node:
            yield from numbers(v)
    elif isinstance(node, (int, float)) and not isinstance(node, bool):
        yield float(node)

payloads = [payload_of(e["prompt"]) for e in witness.inner.transcript.entries]
sent_numbers = [x for p in payloads for x in numbers(p)]

print(f"{'day':>3}  {'ticker':10} {'price shown':>12} {'fair value':>11} "
      f"{'mispricing':>11}   in the prompt?")
leaks = checked = 0
for day, (key, shown) in enumerate(zip(witness.key, payloads)):
    price = {a["symbol"]: a["price"] for a in shown["assets"]}
    for ticker, (value, s) in key.items():
        if math.isnan(s):
            print(f"{day:>3}  {ticker:10} {price[ticker]:>12,.2f} {'--':>11} "
                  f"{'--':>11}   not set until the first close")
            continue
        # The fair value to a hundredth of a cent and s to 1e-6: tighter
        # than any price the payload quotes, loose enough to catch a
        # copied value.
        hit = any(abs(x - value) < 1e-4 or abs(x - s) < 1e-6
                  for x in sent_numbers)
        checked += 1
        leaks += hit
        print(f"{day:>3}  {ticker:10} {price[ticker]:>12,.2f} {value:>11,.2f} "
              f"{s:>+11.4f}   {'LEAKED' if hit else 'absent'}")

sent = json.dumps([e["prompt"] for e in witness.inner.transcript.entries])
names = [w for w in ("fair_value", "mispricing", "attribution", "crowd_lean")
         if w in sent.lower()]
print(f"\nhidden values in the prompts: {leaks} of {checked} checked")
print(f"forbidden field names in the prompts: {names or 'none'}")
print(f"numbers scanned: {len(sent_numbers):,}, prompt bytes: {len(sent):,}")
# Asserted as well as printed, so a leak fails the notebook in the test suite.
assert not leaks and not names, (leaks, names)

day  ticker      price shown  fair value  mispricing   in the prompt?
  0  TECH_A           140.00          --          --   not set until the first close
  0  TECH_B            95.00          --          --   not set until the first close
  0  BANK_A            60.00          --          --   not set until the first close
  0  STAPLE_A          48.00          --          --   not set until the first close
  1  TECH_A           142.97      140.42     +0.0180   absent
  1  TECH_B            93.35       94.53     -0.0126   absent
  1  BANK_A            60.25       59.68     +0.0095   absent
  1  STAPLE_A          48.17       50.10     -0.0394   absent
  2  TECH_A           144.34      141.73     +0.0182   absent
  2  TECH_B            95.34       96.47     -0.0118   absent
  2  BANK_A            61.54       60.93     +0.0099   absent
  2  STAPLE_A          48.69       50.59     -0.0382   absent
  3  TECH_A           147.44      144.75     +0.0184   absent
  3  TECH_B            95.89    

The model priced this book without ever being told what the simulator thinks
the book is worth. That is what makes the comparison mean something: the
agent inferred, it was not informed.

Day 0 has no key to leak: the engine sets `mispricing_s` at the first close,
so the column reads NaN until then.

This cell used to rebuild a fair value from the roster's fundamentals with
`tf.fair_value` and scan the prompt text for it. That is not the number the
engine holds: it gave 99.74 for `TECH_A`, where the engine's fair value on
day 1 was 140.42. At two decimals one of those rebuilt figures also matched
an unrelated number in the prompt, and the scan reported a leak that was
not one. Reading the engine's own values through the declared route checks
the right numbers.

`tests/test_openai_agents.py` proves the same boundary twice on every run --
once against an engine proxy that raises if the forbidden surface is
*touched*, and once by scanning what the model actually received.

## 9. Reproducing this

The recording is committed at `tests/fixtures/openai_agents/five-days.json`.
Replaying it needs no API key, no network, and not even the SDK installed --
the market re-executes for real and only the agent's answers come from the
file, keyed by a digest of the exact input.

Change the observation mapping, the brief or the market and the digest moves,
the key goes missing, and the replay **refuses** naming the step rather than
answering the new question with an answer given to the old one. That is not a
hypothetical: rewriting the brief to name both size limits invalidated the
first recording, and 0.8.5's three changes to the market invalidated three
more, which is how the fixture above came to be re-recorded.

In [11]:
same = tf.evaluate(
    {"pm": OpenAIAgentsAdapter(mode="replay",
                               transcript=Transcript.load(example.FIXTURE))},
    seed=example.SEED, universe=example.universe(), days=example.DAYS)["pm"]

print(f"this notebook   {card.pnl:+,.0f}")
print(f"fresh replay    {same.pnl:+,.0f}")
print(f"identical       {same.pnl == card.pnl}")

# Asserted, not merely printed. `tf.evaluate` scores many agents and so
# ABSORBS a refusal into `card.errors` rather than raising -- correct for a
# harness, exactly wrong for a notebook demonstrating one run. A corrupted
# recording would make some decisions fail to replay, and this notebook
# would print a smaller, plausible, wrong result and every cell would still
# be green. These three lines are what make that impossible.
assert same.pnl == card.pnl, "the replay did not reproduce this run"
assert len(agent.record) == example.DAYS, (
    f"only {len(agent.record)} of {example.DAYS} decisions replayed")
# Refusals are allowed, and only of one kind: the market declining an
# order the book could not fund, which is the agent's sizing and is part of
# what this notebook shows. Any other error is a replay that went wrong.
assert not [e for e in card.errors if "leverage" not in e], card.errors
assert card.rejected == len(card.errors), (card.rejected, card.errors)

if not LIVE:
    print()
    print(f"recorded with   {agent.transcript.meta.get('model')} "
          f"({agent.transcript.meta.get('provider')}) on "
          f"{agent.transcript.meta.get('recorded_utc')}")

this notebook   +57,576
fresh replay    +57,576
identical       True

recorded with   gpt-5.2 (openai) on 2026-09-26T10:29:25+00:00


---

**Where to go next.** `examples/integrations/openai_agents/five_days.py` is the
same experiment as a script, using the SDK's own deterministic model so it
runs offline in seconds. `examples/integrations/finrobot/rate_shock.ipynb`
takes a different shape entirely: a checkpoint, a fork, and one macro
intervention applied to a single arm, which is how you ask what an agent
would have done otherwise.